In [ ]:
# Install required packages
!pip install ultralytics torch torchvision --quiet
!pip install stable-baselines3 gymnasium tensorboard --quiet
!pip install opencv-python numpy matplotlib seaborn --quiet
!pip install pandas scikit-learn --quiet
!pip install tqdm psutil --quiet

# Check installations
import torch
import cv2
import gymnasium as gym
import ultralytics
from stable_baselines3 import PPO

print(f"PyTorch: {torch.__version__}")
print(f"OpenCV: {cv2.__version__}")
print(f"Gymnasium: {gym.__version__}")
print(f"Ultralytics: {ultralytics.__version__}")

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
import os
import torch
from ultralytics import YOLO
import numpy as np
import cv2
from pathlib import Path

# Create directories
directories = ['data', 'models', 'logs', 'sample_images']
for dir_name in directories:
    Path(dir_name).mkdir(exist_ok=True)
    print(f"Created directory: {dir_name}")

# Load YOLO model
print("Loading YOLO model...")
yolo_model = YOLO('yolov8n.pt')
print("YOLO model loaded successfully!")

# Test YOLO model
test_image = np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8)
results = yolo_model.predict(test_image, conf=0.25, verbose=False)
print(f"YOLO test successful! Detected {len(results[0].boxes)} objects in test image.")

In [ ]:
import logging
from typing import List, Dict, Any, Optional
from datetime import datetime
import json

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class DroneNLP:
    """NLP Module that explains drone results in natural language."""

    def __init__(self, language: str = "en"):
        self.language = language
        self.class_names = ["tree", "building", "pole", "person", "vehicle", "aircraft"]
        self.action_names = {
            0: "turn left",
            1: "turn right", 
            2: "climb up",
            3: "descend down",
            4: "move forward",
            5: "stop",
        }
        self.distance_descriptions = {
            "very_close": "very close",
            "close": "close", 
            "medium": "medium distance",
            "far": "far",
            "very_far": "very far"
        }
        
        # Flight history for analysis
        self.flight_history = []
        self.session_start = datetime.now()

    def explain_current_state(self, obstacles: List[Dict], action: int, confidence: float = 0.0) -> str:
        """Generate explanation of drone's current state"""
        
        if not obstacles:
            return f"Path is clear. Drone {self.action_names.get(action, 'taking action')}."
        
        # Get the most dangerous obstacle
        main_obstacle = max(obstacles, key=lambda o: o.get("confidence", 0) * o.get("w", 0) * o.get("h", 0))
        
        object_name = self._get_object_name(main_obstacle)
        object_confidence = int(main_obstacle.get("confidence", 0) * 100)
        action_description = self.action_names.get(action, "taking action")
        
        # Determine obstacle position
        position = self._get_position_description(main_obstacle)
        
        # Determine distance (from box size)
        distance = self._get_distance_description(main_obstacle)
        
        # If multiple obstacles
        additional_info = ""
        if len(obstacles) > 1:
            other_objects = self._get_other_objects_summary(obstacles[1:])
            additional_info = f" Also there are {other_objects}."
        
        # If action confidence is high
        confidence_info = ""
        if confidence > 0:
            confidence_pct = int(confidence * 100)
            confidence_info = f" (action confidence: {confidence_pct}%)"
        
        explanation = (
            f"Drone detected {object_name} {position}, {distance} "
            f"({object_confidence}% confidence). "
            f"Taking action: {action_description}{confidence_info}.{additional_info}"
        )
        
        return explanation

    def _get_object_name(self, obstacle: Dict) -> str:
        """Get object name based on class"""
        class_id = obstacle.get("class", 0)
        if 0 <= class_id < len(self.class_names):
            return self.class_names[class_id]
        return "unknown object"

    def _get_position_description(self, obstacle: Dict) -> str:
        """Determine obstacle position"""
        x = obstacle.get("x", 0.5)
        
        if x < 0.33:
            return "to the left"
        elif x > 0.66:
            return "to the right"
        elif x < 0.45:
            return "slightly left"
        elif x > 0.55:
            return "slightly right"
        else:
            return "ahead"

    def _get_distance_description(self, obstacle: Dict) -> str:
        """Determine distance from box size"""
        width = obstacle.get("w", 0.1)
        height = obstacle.get("h", 0.1)
        size = width * height
        
        if size > 0.3:
            return self.distance_descriptions["very_close"]
        elif size > 0.15:
            return self.distance_descriptions["close"]
        elif size > 0.08:
            return self.distance_descriptions["medium"]
        elif size > 0.04:
            return self.distance_descriptions["far"]
        else:
            return self.distance_descriptions["very_far"]

    def _get_other_objects_summary(self, obstacles: List[Dict]) -> str:
        """Get summary of other obstacles"""
        if not obstacles:
            return ""
        
        object_counts = {}
        for obs in obstacles:
            name = self._get_object_name(obs)
            object_counts[name] = object_counts.get(name, 0) + 1
        
        summary_parts = []
        for name, count in object_counts.items():
            if count == 1:
                summary_parts.append(f"{name} 1")
            else:
                summary_parts.append(f"{name} {count}")
        
        return f"other obstacles {len(obstacles)}: {', '.join(summary_parts)}"

    def generate_flight_summary(self, flight_history: List[Dict]) -> str:
        """Generate summary of entire flight"""
        
        if not flight_history:
            return "No flight data available."
        
        total_steps = len(flight_history)
        avoidance_actions = sum(1 for h in flight_history if h.get("action") not in [4, 5] and h.get("obstacles"))
        crashes = sum(1 for h in flight_history if h.get("crashed", False))
        
        # Analyze detected objects
        object_counts = {}
        for h in flight_history:
            for obs in h.get("obstacles", []):
                name = self._get_object_name(obs)
                object_counts[name] = object_counts.get(name, 0) + 1
        
        # Get most detected objects
        top_objects = sorted(object_counts.items(), key=lambda x: -x[1])[:3]
        top_objects_str = ", ".join(f"{name}({count})" for name, count in top_objects) if top_objects else "none"
        
        # Analyze performance
        crash_rate = (crashes / total_steps * 100) if total_steps > 0 else 0
        avoidance_rate = (avoidance_actions / total_steps * 100) if total_steps > 0 else 0
        
        summary = (
            f"Flight completed. "
            f"Steps {total_steps}, "
            f"obstacles avoided {avoidance_actions} ({avoidance_rate:.1f}%), "
            f"crashes {crashes} ({crash_rate:.1f}%). "
            f"Most detected objects: {top_objects_str}."
        )
        
        return summary

# Initialize NLP module
print("Initializing NLP module...")
nlp = DroneNLP()
print("NLP module initialized successfully!")

# Test NLP module
test_obstacles = [
    {"class": 3, "confidence": 0.9, "x": 0.7, "y": 0.5, "w": 0.3, "h": 0.2},
    {"class": 0, "confidence": 0.7, "x": 0.2, "y": 0.3, "w": 0.1, "h": 0.15}
]

explanation = nlp.explain_current_state(test_obstacles, action=0)
print("NLP Test:")
print(explanation)
print("NLP module working correctly!")

In [ ]:
import gymnasium as gym
import numpy as np
from gymnasium import spaces
from typing import Dict, List, Tuple, Any, Optional

class DroneObstacleEnv(gym.Env):
    """Drone obstacle avoidance environment for reinforcement learning."""

    ACTIONS = {
        0: "turn_left",
        1: "turn_right",
        2: "climb_up",
        3: "descend_down",
        4: "move_forward",
        5: "stop",
    }

    def __init__(self, max_obstacles: int = 10, max_steps: int = 200):
        super().__init__()
        
        # Environment parameters
        self.max_obstacles = max_obstacles
        self.obs_features = 6  # [x, y, w, h, class, confidence]
        self.max_steps = max_steps
        
        # Observation space: normalized coordinates and features
        self.observation_space = spaces.Box(
            low=0.0,
            high=1.0,
            shape=(self.max_obstacles * self.obs_features,),
            dtype=np.float32
        )

        # Action space: 6 discrete actions
        self.action_space = spaces.Discrete(6)

        # Environment state
        self.obstacles = []
        self.step_count = 0
        self.drone_position = [0.5, 0.5]  # Center position
        self.collision_threshold = 0.1
        
        # Statistics
        self.total_reward = 0
        self.episodes_completed = 0
        self.crashes_count = 0

    def reset(self, seed: Optional[int] = None, options: Optional[Dict] = None) -> Tuple[np.ndarray, Dict]:
        """Reset environment to initial state"""
        super().reset(seed=seed)
        
        self.obstacles = self._generate_random_obstacles()
        self.step_count = 0
        self.drone_position = [0.5, 0.5]
        self.total_reward = 0
        
        obs = self._get_observation()
        info = self._get_info()
        
        return obs, info

    def step(self, action: int) -> Tuple[np.ndarray, float, bool, bool, Dict]:
        """Execute one step in the environment"""
        self.step_count += 1
        
        # Calculate reward and termination
        reward, terminated = self._compute_reward(action)
        self.total_reward += reward
        
        # Update drone position based on action
        self._update_drone_position(action)
        
        # Move obstacles (simulate drone movement)
        self._update_obstacles()
        
        # Check for crashes
        crashed = self._check_collision()
        if crashed:
            terminated = True
            self.crashes_count += 1
            reward = -100.0
        
        # Check if episode should truncate
        truncated = self.step_count >= self.max_steps
        
        # Get new observation
        obs = self._get_observation()
        info = self._get_info()
        
        if terminated or truncated:
            self.episodes_completed += 1
        
        return obs, reward, terminated, truncated, info

    def _compute_reward(self, action: int) -> Tuple[float, bool]:
        """Compute reward based on current state and action"""
        danger_obstacles = self._get_danger_obstacles()
        
        if not danger_obstacles:
            # No danger - move forward
            reward = 1.0 if action == 4 else 0.0
            return reward, False
        
        # Danger present - must avoid
        if action in [0, 1, 2, 3]:  # Any avoidance action
            return 10.0, False
        elif action == 4:  # Moving forward into danger
            return -20.0, False
        else:  # Stop
            return -5.0, False

    def _get_danger_obstacles(self) -> List[Dict]:
        """Get obstacles that pose immediate danger"""
        return [
            obs for obs in self.obstacles
            if obs["x"] > 0.4 and obs["w"] > 0.25 and obs["confidence"] > 0.5
        ]

    def _update_drone_position(self, action: int):
        """Update drone position based on action"""
        move_distance = 0.05
        
        if action == 0:  # turn left
            self.drone_position[0] = max(0.1, self.drone_position[0] - move_distance)
        elif action == 1:  # turn right
            self.drone_position[0] = min(0.9, self.drone_position[0] + move_distance)
        elif action == 2:  # climb up
            self.drone_position[1] = max(0.1, self.drone_position[1] - move_distance)
        elif action == 3:  # descend down
            self.drone_position[1] = min(0.9, self.drone_position[1] + move_distance)
        elif action == 4:  # forward (move towards obstacles)
            pass  # Forward movement is simulated by moving obstacles closer

    def _update_obstacles(self):
        """Update obstacle positions (simulate drone forward movement)"""
        obstacle_speed = 0.02
        
        self.obstacles = [
            {**obs, "x": obs["x"] - obstacle_speed}
            for obs in self.obstacles
            if obs["x"] > -0.1  # Remove obstacles that have passed
        ]

    def _check_collision(self) -> bool:
        """Check if drone has collided with any obstacle"""
        for obs in self.obstacles:
            # Simple collision detection based on distance
            distance = np.sqrt((self.drone_position[0] - obs["x"])**2 + 
                             (self.drone_position[1] - obs["y"])**2)
            
            if distance < self.collision_threshold:
                return True
        
        return False

    def _get_observation(self) -> np.ndarray:
        """Get current observation as normalized array"""
        obs_array = np.zeros(self.max_obstacles * self.obs_features, dtype=np.float32)
        
        for i, obs in enumerate(self.obstacles[:self.max_obstacles]):
            base = i * self.obs_features
            obs_array[base + 0] = obs["x"]
            obs_array[base + 1] = obs["y"]
            obs_array[base + 2] = obs["w"]
            obs_array[base + 3] = obs["h"]
            obs_array[base + 4] = obs["class"] / 6.0  # normalize class (0-5)
            obs_array[base + 5] = obs["confidence"]
        
        return obs_array

    def _get_info(self) -> Dict:
        """Get additional environment information"""
        return {
            "step_count": self.step_count,
            "obstacles_count": len(self.obstacles),
            "danger_obstacles": len(self._get_danger_obstacles()),
            "drone_position": self.drone_position.copy(),
            "total_reward": self.total_reward,
            "episodes_completed": self.episodes_completed,
            "crashes_count": self.crashes_count
        }

    def _generate_random_obstacles(self) -> List[Dict]:
        """Generate random obstacles for training"""
        n = np.random.randint(1, 5)
        return [
            {
                "x": np.random.uniform(0.1, 0.9),
                "y": np.random.uniform(0.1, 0.9),
                "w": np.random.uniform(0.05, 0.4),
                "h": np.random.uniform(0.05, 0.4),
                "class": np.random.randint(0, 6),
                "confidence": np.random.uniform(0.5, 1.0),
            }
            for _ in range(n)
        ]

    def obstacles_from_yolo(self, yolo_detections):
        """Update obstacles using actual YOLO detection results"""
        if hasattr(yolo_detections, 'boxes'):
            self.obstacles = [
                {
                    "x": float(box.xywhn[0][0]),
                    "y": float(box.xywhn[0][1]),
                    "w": float(box.xywhn[0][2]),
                    "h": float(box.xywhn[0][3]),
                    "class": int(box.cls),
                    "confidence": float(box.conf),
                }
                for box in yolo_detections.boxes
            ]
        else:
            self.obstacles = []

# Test the environment
print("Testing RL Environment...")
env = DroneObstacleEnv()

# Test reset and step
obs, info = env.reset()
print(f"Environment reset successful!")
print(f"Observation shape: {obs.shape}")
print(f"Initial info: {info}")

# Test a few steps
for step in range(3):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    print(f"Step {step + 1}: Action {action} ({env.ACTIONS[action]}), Reward {reward:.2f}")
    
    if terminated or truncated:
        break

print("RL Environment working correctly!")

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import EvalCallback
import os
import time

print("Starting RL Training...")

# Create vectorized environments for faster training
print("Creating training environments...")
train_env = make_vec_env(DroneObstacleEnv, n_envs=4, seed=42)
eval_env = make_vec_env(DroneObstacleEnv, n_envs=1, seed=123)

print(f"Created {4} training environments and 1 evaluation environment")

# Create PPO model
print("Creating PPO model...")
model = PPO(
    "MlpPolicy",
    train_env,
    verbose=1,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    tensorboard_log="./logs/rl_tensorboard",
)

print("PPO model created successfully!")

# Create evaluation callback
eval_callback = EvalCallback(
    eval_env,
    best_model_save_path="./models/",
    log_path="./logs/evaluations/",
    eval_freq=5000,
    deterministic=True,
    render=False,
    verbose=1
)

# Train the model (short training for demo)
print("Starting training (short demo)...")
start_time = time.time()

try:
    model.learn(
        total_timesteps=20000,  # Short training for demo
        callback=eval_callback,
        progress_bar=True
    )
    
    training_time = time.time() - start_time
    print(f"Training completed in {training_time:.2f} seconds!")
    
    # Save the model
    model.save("./models/drone_ppo_colab")
    print("Model saved to: ./models/drone_ppo_colab")
    
except KeyboardInterrupt:
    print("Training interrupted by user")
except Exception as e:
    print(f"Training failed: {e}")

# Clean up
train_env.close()
eval_env.close()
print("Environment cleanup completed")

In [ ]:
import time
import cv2
import numpy as np
from typing import Dict, List, Optional, Tuple
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

class DroneSystem:
    """Complete drone system: YOLO + RL + NLP"""

    def __init__(self):
        print("Initializing Complete Drone System...")
        
        # Initialize components
        self.yolo_model = yolo_model
        self.env = DroneObstacleEnv()
        self.nlp = DroneNLP()
        
        # Try to load trained RL model
        try:
            if os.path.exists("./models/drone_ppo_colab.zip"):
                self.rl_model = PPO.load("./models/drone_ppo_colab.zip")
                print("Trained RL model loaded!")
            else:
                self.rl_model = None
                print("No trained RL model found, using mock decisions")
        except Exception as e:
            self.rl_model = None
            print(f"Could not load RL model: {e}")
        
        # Statistics
        self.system_stats = {
            "total_frames": 0,
            "detections": 0,
            "avoidance_actions": 0,
            "crashes": 0,
            "start_time": datetime.now(),
        }
        
        self.flight_history = []
        print("Drone System initialized successfully!")

    def process_frame(self, frame: np.ndarray) -> Dict:
        """Process a single frame with complete system"""
        
        # Step 1: YOLO detection
        obstacles, detection_time = self._detect_obstacles(frame)
        
        # Step 2: RL decision
        action, rl_confidence, reasoning = self._make_decision(obstacles)
        
        # Step 3: NLP explanation
        explanation = self.nlp.explain_current_state(obstacles, action, rl_confidence)
        
        # Step 4: Update statistics
        self._update_statistics(obstacles, action, rl_confidence)
        
        # Step 5: Save to history
        flight_event = {
            "timestamp": datetime.now(),
            "obstacles": obstacles,
            "action": action,
            "confidence": rl_confidence,
            "reasoning": reasoning,
            "explanation": explanation,
            "detection_time": detection_time
        }
        self.flight_history.append(flight_event)
        
        # Step 6: Create visualization
        annotated_frame = self._draw_frame(frame, obstacles, action, explanation)
        
        return {
            "obstacles": obstacles,
            "action": action,
            "action_name": DroneObstacleEnv.ACTIONS.get(action, "unknown"),
            "confidence": rl_confidence,
            "explanation": explanation,
            "reasoning": reasoning,
            "frame": annotated_frame,
            "detection_time": detection_time
        }

    def _detect_obstacles(self, frame: np.ndarray) -> Tuple[List[Dict], float]:
        """Detect obstacles using YOLO"""
        start_time = time.time()
        
        try:
            results = self.yolo_model.predict(frame, conf=0.4, verbose=False)
            obstacles = self._extract_obstacles(results[0])
            detection_time = time.time() - start_time
            
            return obstacles, detection_time
            
        except Exception as e:
            print(f"Detection error: {e}")
            return self._mock_detection(frame), time.time() - start_time

    def _extract_obstacles(self, result) -> List[Dict]:
        """Extract obstacles from YOLO results"""
        obstacles = []
        class_names = ["tree", "building", "pole", "person", "vehicle", "aircraft"]
        
        for box in result.boxes:
            try:
                obstacles.append({
                    "class": int(box.cls),
                    "name": class_names[int(box.cls)] if int(box.cls) < len(class_names) else "unknown",
                    "confidence": float(box.conf),
                    "x": float(box.xywhn[0][0]),
                    "y": float(box.xywhn[0][1]),
                    "w": float(box.xywhn[0][2]),
                    "h": float(box.xywhn[0][3])
                })
            except Exception as e:
                continue
        
        return obstacles

    def _mock_detection(self, frame: np.ndarray) -> List[Dict]:
        """Generate mock obstacles for testing"""
        np.random.seed(int(time.time()) % 1000)
        n_obstacles = np.random.randint(0, 4)
        
        obstacles = []
        for i in range(n_obstacles):
            obstacles.append({
                "class": np.random.randint(0, 6),
                "confidence": np.random.uniform(0.5, 1.0),
                "x": np.random.uniform(0.1, 0.9),
                "y": np.random.uniform(0.1, 0.9),
                "w": np.random.uniform(0.05, 0.3),
                "h": np.random.uniform(0.05, 0.3)
            })
        
        return obstacles

    def _make_decision(self, obstacles: List[Dict]) -> Tuple[int, float, Dict]:
        """Make decision using RL or mock decision"""
        if self.rl_model is None:
            return self._mock_decision(obstacles)
        
        try:
            # Update environment with current obstacles
            self.env.obstacles_from_yolo(type('MockResult', (), {'boxes': obstacles})())
            
            # Get observation
            obs = self.env._get_observation()
            
            # Get action from RL model
            action, _states = self.rl_model.predict(obs, deterministic=True)
            action = int(action)
            
            # Calculate confidence
            confidence = min(0.9, 0.5 + len(obstacles) * 0.1)
            
            reasoning = {
                "primary_reason": "avoidance" if obstacles else "safety",
                "confidence": confidence,
                "obstacle_count": len(obstacles)
            }
            
            return action, confidence, reasoning
            
        except Exception as e:
            print(f"RL decision error: {e}")
            return self._mock_decision(obstacles)

    def _mock_decision(self, obstacles: List[Dict]) -> Tuple[int, float, Dict]:
        """Generate mock decision for testing"""
        if not obstacles:
            return 4, 0.8, {"primary_reason": "safety", "confidence": 0.8}
        
        # Simple avoidance logic
        main_obstacle = max(obstacles, key=lambda o: o.get("confidence", 0) * o.get("w", 0))
        x_pos = main_obstacle.get("x", 0.5)
        
        if x_pos < 0.4:
            action = 1  # turn right
        elif x_pos > 0.6:
            action = 0  # turn left
        else:
            action = 2  # climb
        
        confidence = min(0.9, 0.6 + len(obstacles) * 0.1)
        
        reasoning = {
            "primary_reason": "avoidance",
            "confidence": confidence,
            "obstacle_count": len(obstacles),
            "main_obstacle": main_obstacle.get("name", "unknown")
        }
        
        return action, confidence, reasoning

    def _update_statistics(self, obstacles: List[Dict], action: int, confidence: float):
        """Update system statistics"""
        self.system_stats["total_frames"] += 1
        
        if obstacles:
            self.system_stats["detections"] += len(obstacles)
        
        if action in [0, 1, 2, 3]:  # avoidance actions
            self.system_stats["avoidance_actions"] += 1
        
        if obstacles and confidence < 0.3:
            self.system_stats["crashes"] += 1

    def _draw_frame(self, frame: np.ndarray, obstacles: List[Dict], action: int, explanation: str) -> np.ndarray:
        """Draw annotations on frame"""
        annotated = frame.copy()
        
        # Draw obstacles
        for obs in obstacles:
            x, y, w, h = obs["x"], obs["y"], obs["w"], obs["h"]
            img_h, img_w = frame.shape[:2]
            
            # Convert normalized coordinates to pixel coordinates
            x1 = int((x - w/2) * img_w)
            y1 = int((y - h/2) * img_h)
            x2 = int((x + w/2) * img_w)
            y2 = int((y + h/2) * img_h)
            
            # Draw bounding box
            color = (0, 255, 0) if obs["confidence"] > 0.7 else (0, 255, 255)
            cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
            
            # Draw label
            label = f"{obs.get('name', 'unknown')} {obs['confidence']:.2f}"
            cv2.putText(annotated, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
        
        # Draw action info
        action_name = DroneObstacleEnv.ACTIONS.get(action, "unknown")
        cv2.putText(
            annotated, f"Action: {action_name}",
            (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2
        )
        
        # Draw explanation (truncated)
        explanation_short = explanation[:60] + "..." if len(explanation) > 60 else explanation
        cv2.putText(
            annotated, explanation_short,
            (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 1
        )
        
        return annotated

    def run_demo(self, num_frames: int = 10):
        """Run a demo of the complete system"""
        print(f"Running demo with {num_frames} frames...")
        
        results = []
        
        for i in range(num_frames):
            # Generate test frame
            test_frame = np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8)
            
            # Process frame
            result = self.process_frame(test_frame)
            
            # Store result
            results.append(result)
            
            # Print info
            print(f"Frame {i+1}: {result['action_name']} | {result['explanation'][:50]}...")
        
        # Generate final summary
        self._generate_final_summary()
        
        return results

    def _generate_final_summary(self):
        """Generate final flight summary"""
        duration = datetime.now() - self.system_stats["start_time"]
        
        print("\n" + "="*60)
        print("DRONE SYSTEM - FINAL SUMMARY")
        print("="*60)
        print(f"Processing Statistics:")
        print(f"   Total Frames: {self.system_stats['total_frames']}")
        print(f"   Processing Time: {duration.total_seconds():.2f}s")
        print(f"   Average FPS: {self.system_stats['total_frames']/max(1, duration.total_seconds()):.1f}")
        print(f"Detection Statistics:")
        print(f"   Total Detections: {self.system_stats['detections']}")
        print(f"   Detection Rate: {self.system_stats['detections']/max(1, self.system_stats['total_frames']):.2f} per frame")
        print(f"Decision Statistics:")
        print(f"   Avoidance Actions: {self.system_stats['avoidance_actions']}")
        print(f"   Avoidance Rate: {self.system_stats['avoidance_actions']/max(1, self.system_stats['total_frames']):.2%}")
        print(f"   Crashes: {self.system_stats['crashes']}")
        print(f"   Crash Rate: {self.system_stats['crashes']/max(1, self.system_stats['total_frames']):.2%}")
        print("="*60)
        
        # Generate NLP summary
        nlp_summary = self.nlp.generate_flight_summary(self.flight_history)
        print(f"Flight Summary:")
        print(f"   {nlp_summary}")
        print("="*60)

# Initialize and test the complete system
print("Initializing Complete Drone System...")
drone_system = DroneSystem()

# Run demo
demo_results = drone_system.run_demo(num_frames=5)
print("Demo completed successfully!")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from collections import Counter

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

def create_visualizations(results):
    """Create comprehensive visualizations of the drone system performance"""
    
    if not results:
        print("No results to visualize!")
        return
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('Drone System Performance Analysis', fontsize=16, fontweight='bold')
    
    # 1. Actions Distribution
    actions = [r['action_name'] for r in results]
    action_counts = Counter(actions)
    
    axes[0, 0].pie(action_counts.values(), labels=action_counts.keys(), autopct='%1.1f%%', startangle=90)
    axes[0, 0].set_title('Actions Distribution')
    
    # 2. Detection Confidence Distribution
    confidences = []
    for r in results:
        confidences.extend([obs['confidence'] for obs in r['obstacles']])
    
    if confidences:
        axes[0, 1].hist(confidences, bins=20, alpha=0.7, edgecolor='black')
        axes[0, 1].set_title('Detection Confidence Distribution')
        axes[0, 1].set_xlabel('Confidence')
        axes[0, 1].set_ylabel('Frequency')
    else:
        axes[0, 1].text(0.5, 0.5, 'No detections', ha='center', va='center', transform=axes[0, 1].transAxes)
        axes[0, 1].set_title('Detection Confidence Distribution')
    
    # 3. Object Classes Detected
    object_classes = []
    for r in results:
        object_classes.extend([obs.get('name', 'unknown') for obs in r['obstacles']])
    
    if object_classes:
        class_counts = Counter(object_classes)
        axes[0, 2].bar(class_counts.keys(), class_counts.values())
        axes[0, 2].set_title('Object Classes Detected')
        axes[0, 2].set_xlabel('Object Class')
        axes[0, 2].set_ylabel('Count')
        axes[0, 2].tick_params(axis='x', rotation=45)
    else:
        axes[0, 2].text(0.5, 0.5, 'No objects detected', ha='center', va='center', transform=axes[0, 2].transAxes)
        axes[0, 2].set_title('Object Classes Detected')
    
    # 4. Decision Confidence Over Time
    decision_confidences = [r['confidence'] for r in results]
    frames = range(1, len(results) + 1)
    
    axes[1, 0].plot(frames, decision_confidences, marker='o', linewidth=2, markersize=8)
    axes[1, 0].set_title('Decision Confidence Over Time')
    axes[1, 0].set_xlabel('Frame Number')
    axes[1, 0].set_ylabel('Confidence')
    axes[1, 0].grid(True, alpha=0.3)
    
    # 5. Obstacle Count Over Time
    obstacle_counts = [len(r['obstacles']) for r in results]
    
    axes[1, 1].plot(frames, obstacle_counts, marker='s', linewidth=2, markersize=8, color='orange')
    axes[1, 1].set_title('Obstacle Count Over Time')
    axes[1, 1].set_xlabel('Frame Number')
    axes[1, 1].set_ylabel('Number of Obstacles')
    axes[1, 1].grid(True, alpha=0.3)
    
    # 6. Detection Time Performance
    detection_times = [r['detection_time'] * 1000 for r in results]  # Convert to milliseconds
    
    axes[1, 2].bar(frames, detection_times, alpha=0.7, color='green')
    axes[1, 2].set_title('Detection Time Performance')
    axes[1, 2].set_xlabel('Frame Number')
    axes[1, 2].set_ylabel('Detection Time (ms)')
    axes[1, 2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics
    print("\nDetailed Performance Statistics:")
    print("="*50)
    print(f"Total Frames Processed: {len(results)}")
    print(f"Total Detections: {sum(obstacle_counts)}")
    print(f"Average Detections per Frame: {np.mean(obstacle_counts):.2f}")
    print(f"Average Decision Confidence: {np.mean(decision_confidences):.3f}")
    print(f"Average Detection Time: {np.mean(detection_times):.2f} ms")
    print(f"Most Common Action: {action_counts.most_common(1)[0][0]} ({action_counts.most_common(1)[0][1]} times)")
    
    if object_classes:
        most_common_object = Counter(object_classes).most_common(1)[0]
        print(f"Most Detected Object: {most_common_object[0]} ({most_common_object[1]} times)")
    
    print("="*50)

# Create visualizations
print("Creating performance visualizations...")
create_visualizations(demo_results)
print("Visualizations completed!")